# Checkpoint 2 — Fase 1: sanity-check forward + stima velocità (Windows)

Notebook pensato per **Windows + Jupyter**. Importa i moduli `forward_model.py`
e `estimate_velocities.py` (tienili nella **stessa cartella** di questo notebook).

Su Windows il `multiprocessing` usa `spawn`: i worker DEVONO stare nei `.py`
importati (non in celle). Questo notebook usa `ProcessPoolExecutor` con
**fallback automatico a seriale**. Tu scegli la modalità con `MODE`.


In [1]:
import os, sys, time
import numpy as np

# >>> 1) IMPOSTA QUI I PATH LOCALI <<<
# Cartella con i 4 file .npy scaricati dalla chat:
DATA_DIR = r"C:\Users\simon\SCT4AMM\SubstrateFinder"   # <-- MODIFICA
# Cartella dove salvare i risultati (es. velocities.npy):
OUT_DIR  = r"C:\Users\simon\SCT4AMM\SubstrateFinder"   # <-- MODIFICA

# Assicura che i moduli .py nella stessa cartella del notebook siano importabili
sys.path.insert(0, os.getcwd())

import forward_model as fm
import estimate_velocities as ev

# Diciamo ai moduli dove sono i dati (sovrascrive il path /mnt/... di default)
ev.UP = DATA_DIR
print("CPU logici rilevati:", os.cpu_count())
print("Moduli importati. DATA_DIR =", DATA_DIR)


CPU logici rilevati: 16
Moduli importati. DATA_DIR = C:\Users\simon\SCT4AMM\SubstrateFinder


In [2]:
signals  = np.load(os.path.join(DATA_DIR, "checkpoint2dataset.npy"))
aniso    = np.load(os.path.join(DATA_DIR, "checkpoint2anisotropy.npy")).ravel()
focal    = np.load(os.path.join(DATA_DIR, "checkpoint2FocalPoints.npy"))  # gia in [-2,2]^2
starting = np.load(os.path.join(DATA_DIR, "checkpoint2starting.npy"))
print("signals:", signals.shape, "| aniso:", aniso, "| starting shape:", starting.shape)


signals: (5, 20, 501) | aniso: [2. 3. 1. 1. 2.] | starting shape: (5, 2)


## 1) Sanity-check del forward model

Confronta i tempi simulati dal FMM con gli indici-campione osservati nei segnali.
Sono solo 10 risoluzioni FMM (5 scenari × 2 fronti): gira in pochi minuti anche
in seriale. **Verifica che `forward front0` ≈ indici osservati** (es. ~48, 45, 39
per lo scenario 0). Se sì → ordine sensori e scala temporale sono corretti.


In [3]:
obs = ev.load_observations(signals, starting)   # tempi+pesi osservati

for s in range(signals.shape[0]):
    (x0,y0),(x1,y1) = focal[s]
    t0,t1 = starting[s]
    a = aniso[s]
    xc,yc,r = ev.REF_LESION[s]
    tf0, tf1 = fm.forward_activation_times(x0,y0,t0, x1,y1,t1, a, xc,yc,r)
    obs0 = obs[(s,0)][0]
    print(f"Scenario {s}  a_ratio={a}")
    print("  forward front0 (ms) :", np.round(tf0[:5],1))
    print("  osservati front0    :", np.round(obs0[:5],1))
    print()


Scenario 0  a_ratio=2.0
  forward front0 (ms) : [58.4 55.6 47.4 44.6 85.1]
  osservati front0    : [48. 45. 39. 38. 74.]

Scenario 1  a_ratio=3.0
  forward front0 (ms) : [42.9 41.9 27.9 17.8 73. ]
  osservati front0    : [48. 44. 23. 17. 58.]

Scenario 2  a_ratio=1.0
  forward front0 (ms) : [34.6 32.1 25.1 23.  42.3]
  osservati front0    : [30. 28. 22. 20. 36.]

Scenario 3  a_ratio=1.0
  forward front0 (ms) : [45.3 42.7 35.7 33.4 51.9]
  osservati front0    : [40. 38. 33. 33. 46.]

Scenario 4  a_ratio=2.0
  forward front0 (ms) : [28.1 30.2 37.8 40.9 24.6]
  osservati front0    : [22. 24. 30. 32. 20.]



## 2) Stima congiunta delle velocità (v_fast, v_slow)

Scegli `MODE`:
- `"serial"` — semplice e robustissimo su Windows. Più lento.
- `"parallel"` — usa `ProcessPoolExecutor`. Su Windows+Jupyter a volte fallisce:
  in quel caso il codice **ricade automaticamente in seriale**.

`N_WORKERS=None` usa tutti i core logici. Ogni iterazione dell'ottimizzatore
calcola 10 forward FMM; Nelder-Mead ne fa tipicamente 40–80.


In [6]:
MODE = "parallel"          # "serial" oppure "parallel"  <-- SCEGLI TU
N_WORKERS = None          # None = tutti i core logici; oppure un intero

from scipy.optimize import minimize
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x=None, **k: x   # no-op se tqdm non installato

jobs = ev.build_jobs(focal, starting, aniso)

# contatore valutazioni per la progress bar
_state = {"n": 0, "best": np.inf}

def make_executor():
    if MODE != "parallel":
        return None
    try:
        from concurrent.futures import ProcessPoolExecutor
        exe = ProcessPoolExecutor(max_workers=N_WORKERS)
        # test rapido di serializzazione
        _ = list(exe.map(ev._forward_one_packed,
                         [(jobs[0][2], 80.0, 20.0)]))
        print("Parallelo attivo:", exe._max_workers, "worker")
        return exe
    except Exception as e:
        print("Parallelo non disponibile, fallback SERIALE. Motivo:", repr(e))
        return None

def objective_tracked(params, executor):
    val = ev.objective(params, jobs, obs, executor=executor)
    _state["n"] += 1
    if val < _state["best"]:
        _state["best"] = val
    print(f"  eval {_state['n']:3d}  v=({params[0]:7.2f},{params[1]:7.2f})  SSE={val:.3f}",
          end="\r")
    return val

print("MODE =", MODE)


MODE = parallel


In [7]:
executor = make_executor()
t_start = time.time()
try:
    res = minimize(objective_tracked, x0=[80.0, 20.0],
                   args=(executor,),
                   method="Nelder-Mead",
                   options={"xatol": 0.5, "fatol": 1e-2, "disp": True})
finally:
    if executor is not None:
        executor.shutdown()

v_fast, v_slow = res.x
print("\n" + "="*50)
print(f"v_fast = {v_fast:.3f}")
print(f"v_slow = {v_slow:.3f}")
print(f"rapporto v_fast/v_slow = {v_fast/v_slow:.3f}")
print(f"SSE finale = {res.fun:.4f}")
print(f"tempo totale = {time.time()-t_start:.1f} s")
print("="*50)

np.save(os.path.join(OUT_DIR, "velocities.npy"), np.array([v_fast, v_slow]))
print("Salvato velocities.npy in", OUT_DIR)


Parallelo attivo: 16 worker
Optimization terminated successfully.12806.751
         Current function value: 12806.751404
         Iterations: 34
         Function evaluations: 63

v_fast = 65.541
v_slow = 22.885
rapporto v_fast/v_slow = 2.864
SSE finale = 12806.7514
tempo totale = 1288.4 s
Salvato velocities.npy in C:\Users\simon\SCT4AMM\SubstrateFinder


In [4]:
import importlib, verify_velocities
importlib.reload(verify_velocities)
vv = verify_velocities

# usa l'executor parallelo già definito nel notebook (o None per seriale)
from concurrent.futures import ProcessPoolExecutor
executor = ProcessPoolExecutor(max_workers=16)   # o metti None per seriale

# 1) confronto residui tra i set di velocità
velocity_sets = {
    "stimate 65/23": (65.541, 22.885),
    "canoniche 80/20": (80.0, 20.0),
    "80/22.9": (80.0, 22.885),     # v_fast canonica, v_slow stimata
}
rows = vv.verify_residuals(signals, focal, starting, aniso,
                           velocity_sets, executor=executor)

# 2) ri-stima sui soli scenari isotropi (a_ratio=1)
refit, iso_idx = vv.refit_isotropic(signals, focal, starting, aniso,
                                    executor=executor)

executor.shutdown()

# 3) report
vv.report(rows, refit, iso_idx)

RESIDUI (SSE) PER SCENARIO E SET DI VELOCITA'
           set | s     0 s     1 s     2 s     3 s     4 |       TOT
--------------------------------------------------------------------
 stimate 65/23 |  2329.1  1683.2   817.5   836.1  7140.9 |   12806.8
canoniche 80/20 |  4069.3  3825.5  1696.4  2636.4 10613.7 |   22841.4
       80/22.9 |  4149.6  3870.2  1696.4  2636.4 10613.7 |   22966.3

RI-STIMA SUI SOLI SCENARI ISOTROPI [2, 3] (a_ratio=1)
  v_fast = 61.857
  v_slow = 22.768   (poco vincolata se la lesione e' piccola)
  SSE    = 1478.195

  --> Se v_fast qui e' ~80, il valore globale piu' basso e' un
      compromesso sugli scenari anisotropi. Se e' ~66, e' reale.


In [8]:
import numpy as np
X = np.load(r'C:\Users\simon\SCT4AMM\SubstrateFinder\X_train.npy')
y = np.load(r'C:\Users\simon\SCT4AMM\SubstrateFinder\y_train.npy')
print('X shape:', X.shape)          # atteso (50000, 46)
print('y shape:', y.shape)          # atteso (50000, 4)
print('NaN in X:', np.isnan(X).sum())
print('NaN in y:', np.isnan(y).sum())
print()
print('y[:,0] a_ratio : min=', y[:,0].min().round(2), 'max=', y[:,0].max().round(2))  # ~1..9
print('y[:,1] xc      : min=', y[:,1].min().round(2), 'max=', y[:,1].max().round(2))  # ~-1.6..1.6
print('y[:,2] yc      : min=', y[:,2].min().round(2), 'max=', y[:,2].max().round(2))
print('y[:,3] r       : min=', y[:,3].min().round(2), 'max=', y[:,3].max().round(2))  # ~0.05..0.6
print()
print('X tempi front0 (primi 20): min=', X[:,:20].min().round(1), 'max=', X[:,:20].max().round(1))
print('X tempi front1 (secondi 20): min=', X[:,20:40].min().round(1), 'max=', X[:,20:40].max().round(1))

X shape: (50000, 46)
y shape: (50000, 4)
NaN in X: 0
NaN in y: 0

y[:,0] a_ratio : min= 1.0 max= 9.0
y[:,1] xc      : min= -1.6 max= 1.6
y[:,2] yc      : min= -1.6 max= 1.6
y[:,3] r       : min= 0.05 max= 0.6

X tempi front0 (primi 20): min= 1.0 max= 428.0
X tempi front1 (secondi 20): min= 260.4 max= 686.6
